# 01 — Training Colab Runner

Run `notebooks/00_preprocess_and_align.ipynb` first. That notebook owns raw-image preprocessing, alignment, filtering, `manifest.csv`, and the Hugging Face dataset export.

This notebook starts after those artifacts already exist and focuses on optional model training.

## 1. Mount Drive

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
IS_MACOS = sys.platform == "darwin"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

: 

## 2. Clone Or Update Repo

In [ ]:
if IN_COLAB:
    REPO_URL = 'https://github.com/gabeweng/image-style-transfer.git'
    REPO_DIR = '/content/image-style-transfer'

    import os

    if os.path.exists(REPO_DIR):
        %cd {REPO_DIR}
        !git pull
    else:
        %cd /content
        !git clone {REPO_URL} {REPO_DIR}
        %cd {REPO_DIR}
else:
    REPO_DIR = '.'

## 3. Install Runtime Dependencies

In [33]:
if IN_COLAB:
    !pip install -q uv

# xformers is for hardware acceleration with Nvidia drivers, something macos doesn't support
if not IS_MACOS:
    !uv pip install xformers
!uv pip install pillow pandas torch torchvision diffusers==0.27.2 'transformers>=4.38.0,<5' 'huggingface-hub<0.26' accelerate peft datasets safetensors wandb tqdm

Using Python 3.11.5 environment at: /Users/tominekan/Code/image-style-transfer/.venv
Resolved 64 packages in 172ms                                        
⠙ Preparing packages... (0/6)                                                   
⠙ Preparing packages... (0/6)--------------     0 B/11.78 KiB           
⠙ Preparing packages... (0/6)---------- 11.78 KiB/11.78 KiB         
⠙ Preparing packages... (0/6)---------- 11.78 KiB/11.78 KiB         
markupsafe           ------------------------------ 11.78 KiB/11.78 KiB
⠙ Preparing packages... (0/6)--------------     0 B/171.46 KiB          
markupsafe           ------------------------------ 11.78 KiB/11.78 KiB
⠙ Preparing packages... (0/6)-------------- 16.00 KiB/171.46 KiB        
⠙ Preparing packages... (0/6)-------------- 16.00 KiB/171.46 KiB        
⠙ Preparing packages... (0/6)-------------- 32.00 KiB/171.46 KiB        
⠙ Preparing packages... (0/6)-------------- 48.00 KiB/171.46 KiB        
⠙ Preparing packages... (0/6)-------------- 

## 4. Configure Paths

In [ ]:
if IN_COLAB:
    BASE = '/content/drive/My Drive/CIS_5190_group_project'
else:
    # We're likely in the notebooks folder
    BASE = '..'

# Can change to 'manifest.csv' to 'audit_decisions.csv' to work on human approved images
MANIFEST_CSV = f'{BASE}/manifest.csv'
FILTERED_DIR = f'{BASE}/filtered_aligned'
HF_DATASET_DIR = f'{BASE}/hf_dataset'
# Changed /data/hf_dataset_controlnet to hf_dataset
HF_CONTROLNET_DIR = f'{BASE}/hf_dataset'
CHECKPOINT_DIR = f'{BASE}/checkpoints'

print('Manifest:', MANIFEST_CSV)
print('Filtered images:', FILTERED_DIR)
print('HF dataset:', HF_DATASET_DIR)
print('Optional ControlNet dataset:', HF_CONTROLNET_DIR)
print('Checkpoints:', CHECKPOINT_DIR)

Manifest:The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
 ../manifest.csv
Filtered images: ../filtered_aligned
HF dataset: ../hf_dataset
Optional ControlNet dataset: ../data/hf_dataset_controlnet
Checkpoints: ../checkpoints


## 5. Validate Preprocess And Align Outputs

In [35]:
import json
import os
import pandas as pd

assert os.path.exists(MANIFEST_CSV), f'Missing manifest: {MANIFEST_CSV}'
assert os.path.isdir(FILTERED_DIR), f'Missing filtered image directory: {FILTERED_DIR}'
assert os.path.exists(f'{HF_DATASET_DIR}/metadata.jsonl'), f'Missing HF metadata: {HF_DATASET_DIR}/metadata.jsonl'

manifest_df = pd.read_csv(MANIFEST_CSV)
kept_df = manifest_df[manifest_df['status'] == 'kept'].copy() if 'status' in manifest_df.columns else manifest_df.copy()

print(f'Manifest rows: {len(manifest_df)}')
print(f'Kept final images: {len(kept_df)}')
if 'status' in manifest_df.columns:
    print('Status counts:', manifest_df['status'].value_counts().to_dict())
if len(kept_df):
    print('Train/val split:', kept_df['split'].value_counts().to_dict())
    display(kept_df.head())

missing = [p for p in kept_df['file_name'].head(20) if not os.path.exists(os.path.join(BASE, p))]
assert not missing, f'Some kept manifest files were not found: {missing[:5]}'

# Might need to modify this to work with manual auditing (if enough time)
with open(f'{HF_DATASET_DIR}/metadata.jsonl') as f:
    metadata_rows = sum(1 for _ in f)
print(f'HF metadata rows: {metadata_rows}')
print('Preprocess/alignment artifacts are ready for training.')

Manifest rows: 237
Kept final images: 108
Status counts: {'kept': 108, 'duplicate_filtered': 95, 'crop_dropped': 20, 'alignment_failed': 14}
Train/val split: {'train': 89, 'val': 19}


,file_name,location,time_of_day,weather,is_synthetic,caption,source_file,original_file_name,anchor_file,crop_w,crop_h,crop_area_ratio,matches,inliers,inlier_ratio,representative_score,split,status,drop_reason
34,filtered_aligned/34th/34th_night_clear_aligned...,34th,night,clear,False,A photo of 34th at Penn during night clear wea...,34th/night_clear/34th_night_clear.jpg,34th_night_clear.jpg,34th/night_clear/34th_night_clear.jpg,5469.0,3985.0,0.8906,0,0,1.0000,1.000000,train,kept,NaN
35,filtered_aligned/34th/34th_sunset_cloudy_align...,34th,sunset,cloudy,False,A photo of 34th at Penn during sunset cloudy w...,34th/sunset_cloudy/34th_sunset_cloudy.jpg,34th_sunset_cloudy.jpg,34th/night_clear/34th_night_clear.jpg,5469.0,3985.0,0.8906,646,26,0.0402,0.926439,train,kept,NaN
37,filtered_aligned/agh3rd/agh3rd_daytime_clear_a...,agh3rd,daytime,clear,False,A photo of agh3rd at Penn during daytime clear...,agh3rd/daytime_clear/agh3rd_daytime_clear.jpg,agh3rd_morning_clear.jpg,agh3rd/daytime_clear/agh3rd_daytime_clear.jpg,5336.0,3024.0,0.6594,0,0,1.0000,0.997201,train,kept,NaN
40,filtered_aligned/agh3rd/agh3rd_daytime_cloudy_...,agh3rd,daytime,cloudy,False,A photo of agh3rd at Penn during daytime cloud...,agh3rd/daytime_cloudy/agh3rd_daytime_cloudy_1.jpg,agh3rd_day_cloudy_2.jpg,agh3rd/daytime_clear/agh3rd_daytime_clear.jpg,5336.0,3024.0,0.6594,1013,236,0.2330,0.953739,train,kept,NaN
42,filtered_aligned/agh3rd/agh3rd_night_clear_ali...,agh3rd,night,clear,False,A photo of agh3rd at Penn during night clear w...,agh3rd/night_clear/agh3rd_night_clear_3.jpg,agh3rd_night_clear_4.jpg,agh3rd/daytime_clear/agh3rd_daytime_clear.jpg,5336.0,3024.0,0.6594,560,19,0.0339,0.632718,train,kept,NaN


HF metadata rows: 108
Preprocess/alignment artifacts are ready for training.


## 6. GPU Check

Run the remaining training and inference sections on a GPU runtime.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

: 

## 7. Optional W&B Tracking

Set `USE_WANDB = True` if you want diffusion training metrics and checkpoints logged to Weights & Biases. You will be prompted to paste your W&B API key.

In [37]:
USE_WANDB = False
WANDB_PROJECT = 'image-style-transfer'

if USE_WANDB:
    import os
    os.environ['WANDB_PROJECT'] = WANDB_PROJECT
    os.environ['WANDB_LOG_MODEL'] = 'checkpoint'
    !wandb login
    REPORT_TO_ARG = '--report_to=wandb'
else:
    REPORT_TO_ARG = ''

print('W&B enabled:', USE_WANDB)

W&B enabled: False


## 8. Install Diffusers Training Examples

The LoRA and ControlNet training scripts live in the Hugging Face `diffusers` examples directory.

In [ ]:
if IN_COLAB:
    DIFFUSERS_DIR = '/content/diffusers'
else:
    DIFFUSERS_DIR = '../diffusers'

if os.path.exists(DIFFUSERS_DIR):
    %cd {DIFFUSERS_DIR}
    !git fetch --tags
    !git checkout v0.27.2
else:
    %cd /content
    !git clone --branch v0.27.2 --depth 1 https://github.com/huggingface/diffusers.git {DIFFUSERS_DIR}

%cd {REPO_DIR}

/Users/tominekan/Code/image-style-transfer/diffusers
remote: Enumerating objects: 81497, done.
remote: Counting objects: 100% (81497/81497), done.
remote: Compressing objects: 100% (16006/16006), done.
remote: Total 80688 (delta 62278), reused 78576 (delta 60362), pack-reused 0 (from 0)
Receiving objects: 100% (80688/80688), 71.35 MiB | 32.05 MiB/s, done.
Resolving deltas: 100% (62278/62278), completed with 642 local objects.
From https://github.com/huggingface/diffusers
 * [new tag]             0.1.0       -> 0.1.0
 * [new tag]             0.1.1       -> 0.1.1
 * [new tag]             0.1.2       -> 0.1.2
 * [new tag]             0.1.3       -> 0.1.3
 * [new tag]             v0.0.2      -> v0.0.2
 * [new tag]             v0.0.3      -> v0.0.3
 * [new tag]             v0.0.4      -> v0.0.4
 * [new tag]             v0.10.0     -> v0.10.0
 * [new tag]             v0.10.1     -> v0.10.1
 * [new tag]             v0.10.2     -> v0.10.2
 * [new tag]             v0.11.0     -> v0.11.0
 * [new

## 9. Configure Accelerate

This writes a basic single-GPU config so `accelerate launch` can run without the interactive setup prompt.

In [39]:
!accelerate config default

The history saving thread hit an unexpected error (OperationalError('no such table: history')).History will not be written to the database.
Configuration already exists at /Users/tominekan/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


## 10. Train Stable Diffusion LoRA

This writes LoRA weights to `checkpoints/lora`. It saves frequent checkpoints and resumes from the latest checkpoint if Colab disconnects. The Hugging Face training script includes tqdm progress bars by default.

In [41]:
LORA_DIR = f'{BASE}/checkpoints/lora'

!accelerate launch $DIFFUSERS_DIR/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="$HF_DATASET_DIR" \
  --output_dir="$LORA_DIR" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=10 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --checkpointing_steps=100 \
  --checkpoints_total_limit=3 \
  --resume_from_checkpoint="latest" \
  --caption_column="text" \
  $REPORT_TO_ARG

The history saving thread hit an unexpected error (OperationalError('no such table: history')).History will not be written to the database.
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_cpu_threads_per_process` was set to `8` to improve out-of-box performance when training on CPUs
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/Users/tominekan/Code/image-style-transfer/.venv/lib/python3.11/site-packages/accelerate/accelerator.py:528: UserWarning: `log_with=tensorboard` was passed but no supported trackers are currently installed.
  warnings.warn(f"`log_with={log_with}` was passed but no supported trackers are currently installed.")
05/05/2026 13:58:27 - INFO - __main__ - [RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cpu

Mixed precision type: fp16

/Users/tominekan/Code/image-style-transfer/.venv/lib/python

## 11. Train ControlNet

This writes a fine-tuned ControlNet checkpoint to `checkpoints/controlnet`. It saves frequent checkpoints and resumes from the latest checkpoint if Colab disconnects. This is usually more expensive than LoRA training.

In [ ]:
CONTROLNET_DIR = f'{BASE}/checkpoints/controlnet'

!accelerate launch $DIFFUSERS_DIR/examples/controlnet/train_controlnet.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --output_dir="$CONTROLNET_DIR" \
  --train_data_dir="$HF_CONTROLNET_DIR" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=2 \
  --num_train_epochs=5 \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --checkpointing_steps=100 \
  --checkpoints_total_limit=3 \
  --resume_from_checkpoint="latest" \
  --conditioning_image_column="conditioning_images" \
  --image_column="image" \
  --caption_column="text" \
  $REPORT_TO_ARG

## 12. Final Training Artifacts

In [ ]:
expected_outputs = [
    f'{HF_DATASET_DIR}/metadata.jsonl',
    f'{CHECKPOINT_DIR}/lora',
]

if os.path.exists(HF_CONTROLNET_DIR):
    expected_outputs.append(f'{CHECKPOINT_DIR}/controlnet')

for path in expected_outputs:
    print(('OK     ' if os.path.exists(path) else 'MISSING'), path)